In [ ]:
# 1. Import required modules

from anomalib.data import MVTecAD
from anomalib.engine import Engine

from anomalib.models import Fastflow
from anomalib.models import Padim

import matplotlib.pyplot as plt


from anomalib.data.datasets.base import AnomalibDataset
from pathlib import Path
import pandas as pd

from anomalib.data.utils import LabelName

import torch
from anomalib.metrics.evaluator import Evaluator

torch.serialization.add_safe_globals([Evaluator])



ModuleNotFoundError: No module named 'anomalib'

In [ ]:



class CustomDataset(AnomalibDataset):
    """Custom dataset implementation."""

    def __init__(
        self,
        root: Path | str = "./datasets/robotV2",
        category: str = "default",
        transform = None,
        split = None,
    ):
        super().__init__()

        # Set up dataset
        self.root = Path(root)
        self.category = category
        self.split = split

        # Create samples DataFrame
        self.samples = self._make_dataset()

    def _make_dataset(self) -> pd.DataFrame:
        """Create dataset samples DataFrame."""
        samples_list = []

        
        if self.split == "train":
        
            # Collect normal samples
            normal_path = self.root / "train/good"
            images = normal_path.glob("*.png")
            for image_path in images:
                samples_list.append({
                    "image_path": str(image_path),
                    "gt_label": 0,
                    "label_index": LabelName.NORMAL,
                    "split": "train",
                    "mask_path" : None
                })


        if self.split == "test":
            # Collect anomalous samples
            anomaly_path = self.root / "test/anomaly"
            anomaly_images = anomaly_path.glob("*.png")
            for image_path in anomaly_images:
                #mask_path = anomaly_path / "masks" / f"{image_path.stem}_mask.png"
                samples_list.append({
                    "image_path": str(image_path),
                    "split": "test",
                    "label_index": LabelName.ABNORMAL,
                    "gt_label" : 1,
                    "mask_path" : None
                    
                })
                        # Collect normal samples
            normal_path = self.root / "test/good"
            images = normal_path.glob("*.png")
            for image_path in images:
                samples_list.append({
                    "image_path": str(image_path), 
                    "split": "test",
                    "label_index": LabelName.NORMAL,
                    "gt_label" : 0,
                    "mask_path" : None
                })    
                


        # Create DataFrame
        samples = pd.DataFrame(samples_list)
        samples.attrs["task"] = "classification"
        return samples
        
train_ds = CustomDataset(root="./datasets/robotV2",split="train")
test_ds  = CustomDataset(root="./datasets/robotV2", split="test")

In [ ]:
from torch.utils.data import DataLoader


test_dataloader = DataLoader(
    dataset=test_ds,
    batch_size=32,
    shuffle=True,
    num_workers=0,  # en Windows mejor 0
    collate_fn= test_ds.collate_fn 
)

In [ ]:

from anomalib.data import AnomalibDataModule

 
class CustomDataModule(AnomalibDataModule):

    def __init__(self, train_dataset, test_dataset ,**kwargs):

        super().__init__(**kwargs)

        self._train_dataset = train_dataset

        self._test_dataset = test_dataset
        
    def _setup(self,stage = None):

        self.train_data = self._train_dataset

        self.val_data = self._test_dataset
        
        self.test_data = self._test_dataset


datamodule = CustomDataModule(
    train_dataset=train_ds,
    test_dataset= test_ds,
    train_batch_size=32,
    eval_batch_size=1,
    num_workers=0,
    

)

In [ ]:

model = Padim.load_from_checkpoint("./results/Padim/CustomDataModule/latest/weights/lightning/model.ckpt")

"""
model = Padim(
    backbone="wide_resnet50_2",
    layers=["layer1", "layer2", "layer3"],
    n_features=100,
    
)
"""
"""model = Fastflow(
    backbone="resnet18",
    evaluator=evaluator
)"""

engine = Engine(
    accelerator="gpu",
    max_epochs= 1,
)


In [ ]:

engine.fit(model=model,datamodule=datamodule)

In [ ]:
engine.test(model=model,dataloaders=test_dataloader)

In [ ]:
# Engine prediction step (new cell at index 7)



predictions = engine.predict(
    dataset= test_ds,
    model = model
)


Image Visualization

In [ ]:
from anomalib.visualization.image import visualize_image_item

def process_batch(batch_items):
    visualizations = []
    for item in batch_items:
        vis = visualize_image_item(
            item,
            fields=["image", "anomaly_map"],
            fields_config={"anomaly_map": {"normalize": False}}
        )
        score = item.pred_score.item()
        label = item.pred_label.item()
        visualizations.append({
            "vis":   vis,
            "score": score,
            "label": label,
            "path":  item.image_path,
        })
    return visualizations

def process_predictions(predictions):
    visualizations = []
    for batches in predictions:
        visualizations.extend(process_batch(batches))
    return visualizations


def save_predictions(visualizations, output_dir="./results"):
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    for v in visualizations:
        name = Path(v["path"]).stem
        v["vis"].save(f"{output_dir}/{name}.png")
        





In [ ]:
visualization = process_predictions(predictions)


In [ ]:

from anomalib.metrics import AUROC
import torch

auroc = AUROC(fields=["pred_score", "gt_label"])

# Recorre todos los batches e items
for batch in predictions:
    for item in batch:
        auroc.update(item)

result = auroc.compute()
print(f"AUROC: {result:.4f}")

In [ ]:
import torch
from torchmetrics.classification import BinaryConfusionMatrix
import matplotlib.pyplot as plt
import numpy as np

# Recoge predicciones y etiquetas reales
pred_labels = []
gt_labels   = []

for batch in predictions:
    for item in batch:
        pred_labels.append(item.pred_label.item())
        gt_labels.append(item.gt_label.item())

pred_labels = torch.tensor(pred_labels)
gt_labels   = torch.tensor(gt_labels)

# Calcula la confusion matrix
matrix = BinaryConfusionMatrix( )
matrix.update(pred_labels, gt_labels)
figure = matrix.plot()
plt.show()

In [ ]:

# Visualiza
tn, fp, fn, tp = matrix.flatten().tolist()

fig, ax = plt.subplots(figsize=(6, 5))
data = np.array([[tn, fp], [fn, tp]])

im = ax.imshow(data, cmap="Blues")

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Normal", "Anomalía"])
ax.set_yticklabels(["Normal", "Anomalía"])
ax.set_xlabel("Predicción")
ax.set_ylabel("Real")
ax.set_title("Confusion Matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, data[i, j], ha="center", va="center",
                color="white" if data[i, j] > data.max() / 2 else "black",
                fontsize=14, fontweight="bold")

labels = [["TN", "FP"], ["FN", "TP"]]
for i in range(2):
    for j in range(2):
        ax.text(j, i + 0.3, labels[i][j], ha="center", va="center",
                color="white" if data[i, j] > data.max() / 2 else "black",
                fontsize=10)

plt.colorbar(im)
plt.tight_layout()
plt.show()

print(f"Verdaderos Negativos (TN): {tn}")
print(f"Falsos Positivos     (FP): {fp}")
print(f"Falsos Negativos     (FN): {fn}")
print(f"Verdaderos Positivos (TP): {tp}")
print(f"\nPrecisión:  {tp / (tp + fp):.4f}")
print(f"Recall:     {tp / (tp + fn):.4f}")
print(f"F1Score:    {2*tp / (2*tp + fp + fn):.4f}")

In [ ]:
# Encuentra el falso negativo (anómala clasificada como normal)
for batch in predictions:
    for item in batch:
        gt   = item.gt_label.item()
        pred = item.pred_label.item()
        
        if gt == 1 and pred == 0:  # ← anómala clasificada como normal
            print(f"Path:  {item.image_path}")
            print(f"Score: {item.pred_score.item():.4f}")
            
            vis = visualize_image_item(
                item,
                fields=["image", "anomaly_map"],
                fields_config={"anomaly_map": {"normalize": True}}
            )
            vis.show()

In [ ]:
from torchmetrics.classification import (
    BinaryAUROC,
    BinaryF1Score,
    BinaryPrecision,
    BinaryRecall,
    BinaryConfusionMatrix,
    BinaryROC,
    BinaryPrecisionRecallCurve,
)

# Recoge scores y labels
scores = torch.tensor([item.pred_score.item() for batch in predictions for item in batch])
labels = torch.tensor([item.gt_label.item()   for batch in predictions for item in batch])
preds  = torch.tensor([item.pred_label.item() for batch in predictions for item in batch])

# Calcula todas las métricas
print(f"AUROC:     {BinaryAUROC()(scores, labels):.4f}")
print(f"F1Score:   {BinaryF1Score()(preds, labels):.4f}")
print(f"Precision: {BinaryPrecision()(preds, labels):.4f}")
print(f"Recall:    {BinaryRecall()(preds, labels):.4f}")